# 08 — AI Business Insights with Groq

## Purpose

This notebook adds the generative-AI layer to the e-commerce intelligence platform.

The goal is **not** to let an LLM calculate metrics itself.

Instead:

```text
SQL / Python analytics
        ↓
Trusted business metrics
        ↓
Customer / segment / CLV / retention context
        ↓
Groq LLM
        ↓
Natural-language business insight
```

The LLM is therefore used as an **interpretation and communication layer**, while the numerical calculations remain deterministic.

## Business questions

The AI layer should help answer questions such as:

- What are the most important customer segments?
- Which segments deserve retention attention?
- Which customers have high predicted future value and high inactivity?
- What are the strongest drivers behind predicted customer value?
- What actions should the business prioritize?
- What should management investigate next?

### Guardrail

The prompt explicitly instructs the model to use only the supplied metrics and to avoid inventing numbers.

In [ ]:
from pathlib import Path
import json
import os
import sys
import warnings

import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

RECOMMENDATIONS_PATH = (
    PROCESSED_DIR / "retention_recommendations.csv"
)
CLV_PATH = (
    PROCESSED_DIR / "customer_clv_predictions.csv"
)
SHAP_PATH = (
    PROCESSED_DIR / "clv_shap_global_importance.csv"
)
SEGMENTS_PATH = (
    PROCESSED_DIR / "customer_segments.csv"
)

print(f"Project root: {PROJECT_ROOT}")

## 1. Load analytical outputs

In [ ]:
required_paths = [
    RECOMMENDATIONS_PATH,
    CLV_PATH,
    SHAP_PATH,
    SEGMENTS_PATH,
]

missing = [
    path
    for path in required_paths
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing required artifacts:\n"
        + "\n".join(str(path) for path in missing)
        + "\nRun the preceding notebooks first."
    )

recommendations = pd.read_csv(RECOMMENDATIONS_PATH)
clv = pd.read_csv(CLV_PATH)
shap_global = pd.read_csv(SHAP_PATH)
segments = pd.read_csv(SEGMENTS_PATH)

print("Recommendations:", recommendations.shape)
print("CLV predictions:", clv.shape)
print("SHAP global:", shap_global.shape)
print("Segments:", segments.shape)

## 2. Build a trusted business summary

The LLM should receive compact, calculated information rather than raw transactional tables.

This reduces:

- token usage,
- hallucination opportunities,
- accidental recalculation,
- unnecessary exposure of raw records.

In [ ]:
summary = {}

summary["customer_count"] = int(
    recommendations["customer_unique_id"].nunique()
)

summary["predicted_future_revenue_total"] = float(
    recommendations["predicted_future_revenue"].sum()
)

summary["predicted_future_revenue_average"] = float(
    recommendations["predicted_future_revenue"].mean()
)

summary["average_recency_days"] = float(
    recommendations["recency_days"].mean()
)

summary["average_order_count"] = float(
    recommendations["order_count"].mean()
)

summary["retention_priority_distribution"] = (
    recommendations["retention_priority"]
    .value_counts()
    .to_dict()
)

summary["recommended_action_distribution"] = (
    recommendations["recommended_action"]
    .value_counts()
    .to_dict()
)

if "segment" in recommendations.columns:
    summary["segment_distribution"] = (
        recommendations["segment"]
        .value_counts()
        .to_dict()
    )

summary["top_shap_features"] = (
    shap_global
    .sort_values("mean_abs_shap", ascending=False)
    .head(10)
    .to_dict(orient="records")
)

summary["top_predicted_customers"] = (
    recommendations
    .sort_values(
        "predicted_future_revenue",
        ascending=False,
    )
    .head(10)[
        [
            "customer_unique_id",
            "predicted_future_revenue",
            "recency_days",
            "order_count",
            "retention_priority",
            "recommended_action",
        ]
    ]
    .to_dict(orient="records")
)

print(json.dumps(summary, indent=2, default=str))

## 3. Prepare the Groq client

The API key must be supplied through an environment variable.

### Windows PowerShell

```powershell
$env:GROQ_API_KEY="your_api_key"
```

### Do not

- hardcode the key in this notebook,
- commit the key to Git,
- put the key in Streamlit source code.

The application layer will later use the same environment-variable approach.

In [ ]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    print(
        "GROQ_API_KEY is not set. "
        "The analytical preparation cells can still run, "
        "but the live Groq call will be skipped."
    )
else:
    print("GROQ_API_KEY detected.")

## 4. Create a constrained business-insight prompt

The prompt separates:

### System instructions

How the model should behave.

### Business context

The numerical information calculated by Python.

### Requested output

A concise management-oriented report.

The model is explicitly told:

- do not invent metrics,
- do not claim causality,
- identify uncertainty,
- use supplied numbers,
- distinguish recommendations from observed facts.

In [ ]:
SYSTEM_PROMPT = """
You are an e-commerce business intelligence analyst.

Your job is to interpret trusted analytical metrics supplied by
a Python data pipeline.

Rules:
1. Use only the metrics supplied in the user context.
2. Never invent, estimate, or fabricate numbers.
3. Do not claim that correlations are causal.
4. Clearly distinguish observed metrics from recommendations.
5. Prioritize commercially meaningful insights.
6. Mention uncertainty when the data does not support a strong conclusion.
7. Keep the response concise and executive-friendly.

Return the following sections:
1. Executive Summary
2. Key Customer Insights
3. Retention Priorities
4. Recommended Actions
5. Risks / Caveats
6. What to Investigate Next
"""


def build_business_prompt(summary_data: dict) -> str:
    context = json.dumps(
        summary_data,
        indent=2,
        default=str,
    )

    return f"""
Analyze the following trusted e-commerce analytics context.

Do not calculate new metrics from raw data.
Do not invent missing values.

TRUSTED ANALYTICS CONTEXT:
{context}

Produce an executive-friendly business intelligence report using
the requested sections.
"""


USER_PROMPT = build_business_prompt(summary)

print(USER_PROMPT[:5000])

## 5. Call Groq

The notebook uses the official Groq Python client.

The model name is kept in an environment variable so the project can
change models without modifying the business-logic code.

Recommended configuration:

```powershell
$env:GROQ_MODEL="llama-3.3-70b-versatile"
```

If no model variable is supplied, the notebook uses the same model name as the default.

In [ ]:
if GROQ_API_KEY:
    try:
        from groq import Groq

        groq_client = Groq(
            api_key=GROQ_API_KEY,
        )

        GROQ_MODEL = os.getenv(
            "GROQ_MODEL",
            "llama-3.3-70b-versatile",
        )

        response = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            temperature=0.2,
            messages=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT,
                },
                {
                    "role": "user",
                    "content": USER_PROMPT,
                },
            ],
        )

        ai_report = response.choices[0].message.content

        print(ai_report)

    except ImportError:
        ai_report = None
        print(
            "Groq package is not installed. "
            "Install it with: pip install groq"
        )

    except Exception as exc:
        ai_report = None
        print(f"Groq request failed: {exc}")

else:
    ai_report = None
    print(
        "Live AI generation skipped because GROQ_API_KEY "
        "is not configured."
    )

## 6. Customer-level AI insight

The same pattern can be used for one customer.

The important design choice is that the LLM receives:

- predicted future value,
- recency,
- frequency,
- priority,
- recommendation,
- SHAP drivers.

It does not receive permission to make up customer history.

In [ ]:
# Load detailed SHAP values when available.
CUSTOMER_SHAP_PATH = (
    PROCESSED_DIR / "customer_shap_values.csv"
)

if CUSTOMER_SHAP_PATH.exists():
    customer_shap = pd.read_csv(
        CUSTOMER_SHAP_PATH
    )
else:
    customer_shap = pd.DataFrame()

print(
    "Customer SHAP table:",
    customer_shap.shape,
)

In [ ]:
def get_customer_context(
    customer_id: str,
) -> dict:
    """Return trusted analytical context for one customer."""

    matches = recommendations[
        recommendations["customer_unique_id"].astype(str)
        == str(customer_id)
    ]

    if matches.empty:
        raise ValueError(
            f"Customer {customer_id} not found."
        )

    row = matches.iloc[0]

    context = {
        "customer_unique_id": str(
            row["customer_unique_id"]
        ),
        "predicted_future_revenue": float(
            row["predicted_future_revenue"]
        ),
        "recency_days": float(
            row["recency_days"]
        ),
        "order_count": float(
            row["order_count"]
        ),
        "total_revenue": float(
            row["total_revenue"]
        ),
        "retention_priority": str(
            row["retention_priority"]
        ),
        "recommended_action": str(
            row["recommended_action"]
        ),
        "recommendation_reason": str(
            row["recommendation_reason"]
        ),
    }

    if "segment" in row.index:
        context["segment"] = str(row["segment"])

    return context


example_customer = str(
    recommendations.iloc[0]["customer_unique_id"]
)

example_context = get_customer_context(
    example_customer
)

print(
    json.dumps(
        example_context,
        indent=2,
        default=str,
    )
)

In [ ]:
def build_customer_prompt(
    customer_context: dict,
) -> str:
    """Create a constrained customer insight prompt."""

    context = json.dumps(
        customer_context,
        indent=2,
        default=str,
    )

    return f"""
Create a concise customer-level retention insight.

Use only the supplied customer metrics.

Customer context:
{context}

Return:
1. Customer Status
2. Why This Customer Matters
3. Recommended Action
4. Reasoning
5. Caveat

Do not invent purchase categories, preferences, demographics,
or behavior that is not supplied.
"""


if GROQ_API_KEY and ai_report is not None:
    try:
        customer_response = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            temperature=0.2,
            messages=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT,
                },
                {
                    "role": "user",
                    "content": build_customer_prompt(
                        example_context
                    ),
                },
            ],
        )

        customer_ai_insight = (
            customer_response
            .choices[0]
            .message
            .content
        )

        print(customer_ai_insight)

    except Exception as exc:
        customer_ai_insight = None
        print(f"Customer insight generation failed: {exc}")
else:
    customer_ai_insight = None
    print(
        "Customer AI insight skipped because a live Groq "
        "configuration is unavailable."
    )

## 7. Save the executive AI report

The generated report is saved locally for inspection.

The report is intentionally kept separate from the raw analytical datasets.

This makes it clear which files contain:

- source analytics,
- model outputs,
- generated language.

In [ ]:
REPORT_PATH = (
    PROCESSED_DIR / "ai_business_insights.txt"
)

if ai_report:
    REPORT_PATH.write_text(
        ai_report,
        encoding="utf-8",
    )

    print(f"Saved AI report: {REPORT_PATH}")
else:
    print(
        "No AI report was generated. "
        "Set GROQ_API_KEY and rerun the Groq cell."
    )

# Final validation

Expected optional artifact after a successful Groq call:

```text
data/processed/
└── ai_business_insights.txt
```

### Architecture achieved

```text
Deterministic analytics
        ↓
ML predictions
        ↓
SHAP explanations
        ↓
Business rules
        ↓
Groq
        ↓
Natural-language intelligence
```

The LLM is therefore **not the source of truth** for the project's numbers.

That separation is important for a production-oriented portfolio project and makes the AI layer easier to test, audit, and replace.